In [23]:
import torch
import flh
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn

from transformers import LlamaForCausalLM, AutoTokenizer
from flh.quantized_model.modeling_llama import FLH_FP16LlamaForCausalLM, FLH_LlamaForCausalLM

In [24]:
model_path = "/home/mashaobo/.cache/modelscope/hub/models/LLM-Research/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = LlamaForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [25]:
flh_fp16_model = FLH_FP16LlamaForCausalLM.from_float(model, target_device="cuda", fuse_layernorm=True)

  Creating FLH model structure on CPU...
Copying weights from original model to FLH model (on CPU)...
  - Copying embedding and lm_head...
  - Copying and fusing LayerNorm weights in 16 transformer layers...


  Copying layers: 100%|██████████| 16/16 [00:00<00:00, 55.62it/s]


✓ Weight copying completed!
  Moving model to cuda...
✓ Model successfully moved to cuda!


In [26]:
flh_model = FLH_LlamaForCausalLM.from_float(
    model,
    target_device="cuda",
    weight_bits=16,
    act_bits=16,
    weight_group_size=128,
    act_group_size=128,
    weight_sym=True,
    act_sym=False,
)

  Creating FLH quantized model structure on CPU...
  Quantization config: W16G128_sym / A16G128_asym
Copying and quantizing weights from original model to FLH model (on CPU, FP16)...
  - Copying embedding and lm_head...
  - Copying and quantizing 16 transformer layers...


  Quantizing layers: 100%|██████████| 16/16 [00:07<00:00,  2.25it/s]



✓ Weight copying and quantization completed!

  Moving quantized model to cuda...
✓ Model successfully moved to cuda (FP16)!


In [31]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
    (rotary_emb): Ll

In [27]:
print(model.model.layers[0].input_layernorm.weight)
print(flh_fp16_model.model.layers[0].input_layernorm.weight)

input_layernorm_weight = model.model.layers[0].input_layernorm.weight
w_q = model.model.layers[0].self_attn.q_proj.weight
w_k = model.model.layers[0].self_attn.k_proj.weight
w_v = model.model.layers[0].self_attn.v_proj.weight

w_q_norm = w_q * input_layernorm_weight
w_k_norm = w_k * input_layernorm_weight
w_v_norm = w_v * input_layernorm_weight

print(w_q_norm)

print(flh_fp16_model.model.layers[0].self_attn.q_proj.weight)
print(flh.nn.fast_hadamard_transform(flh_model.model.layers[0].self_attn.q_proj.weight, group_size=128, normalize=True))

Parameter containing:
tensor([0.1582, 0.1807, 0.2695,  ..., 0.2217, 0.2109, 0.1523], device='cuda:0',
       dtype=torch.float16, requires_grad=True)
Parameter containing:
tensor([1., 1., 1.,  ..., 1., 1., 1.], device='cuda:0', dtype=torch.float16,
       requires_grad=True)
tensor([[-0.0029,  0.0013,  0.0059,  ..., -0.0015, -0.0019,  0.0023],
        [ 0.0018,  0.0107,  0.0170,  ..., -0.0074, -0.0031,  0.0009],
        [ 0.0029,  0.0025,  0.0097,  ..., -0.0096, -0.0082, -0.0036],
        ...,
        [ 0.0048,  0.0052,  0.0216,  ..., -0.0170, -0.0066, -0.0051],
        [ 0.0038, -0.0059,  0.0099,  ..., -0.0027, -0.0057, -0.0023],
        [-0.0042, -0.0090, -0.0057,  ...,  0.0133,  0.0027, -0.0001]],
       device='cuda:0', dtype=torch.float16, grad_fn=<MulBackward0>)
Parameter containing:
tensor([[-0.0029,  0.0013,  0.0059,  ..., -0.0015, -0.0019,  0.0023],
        [ 0.0018,  0.0107,  0.0170,  ..., -0.0074, -0.0031,  0.0009],
        [ 0.0029,  0.0025,  0.0097,  ..., -0.0096, -0.0082,

In [28]:
o_proj_weight = model.model.layers[0].self_attn.o_proj.weight
o_proj_fp16_weight = flh_fp16_model.model.layers[0].self_attn.o_proj.weight

print(o_proj_weight[0])
print(o_proj_fp16_weight[0])

o_proj_quant = flh_model.model.layers[0].self_attn.o_proj.weight
o_proj_quant = flh.nn.fast_hadamard_transform(o_proj_quant, group_size=128, normalize=True)
o_proj_quant = flh.nn.fast_hadamard_transform(o_proj_quant.t(), group_size=128, normalize=False).t()
print(o_proj_quant[0])


tensor([ 0.0003,  0.0065,  0.0126,  ..., -0.0078, -0.0015,  0.0114],
       device='cuda:0', dtype=torch.float16, grad_fn=<SelectBackward0>)
tensor([ 0.0003,  0.0065,  0.0126,  ..., -0.0078, -0.0015,  0.0114],
       device='cuda:0', dtype=torch.float16, grad_fn=<SelectBackward0>)
tensor([ 0.0003,  0.0065,  0.0126,  ..., -0.0078, -0.0015,  0.0114],
       device='cuda:0', dtype=torch.float16)


In [30]:
post_attention_layernorm = model.model.layers[0].post_attention_layernorm.weight
post_attention_layernorm_fp16 = flh_fp16_model.model.layers[0].post_attention_layernorm.weight

print(post_attention_layernorm)
print(post_attention_layernorm_fp16)

up_proj_weight = model.model.layers[0].mlp.up_proj.weight
up_proj_weight_fp16 = flh_fp16_model.model.layers[0].mlp.up_proj.weight

print(up_proj_weight * post_attention_layernorm)
print(up_proj_weight_fp16)

up_proj_weight_quant = flh_model.model.layers[0].mlp.up_proj.weight

print(flh.nn.fast_hadamard_transform(up_proj_weight_quant, group_size=128, normalize=True))

Parameter containing:
tensor([0.2041, 0.1992, 0.1846,  ..., 0.2139, 0.2021, 0.2080], device='cuda:0',
       dtype=torch.float16, requires_grad=True)
Parameter containing:
tensor([1., 1., 1.,  ..., 1., 1., 1.], device='cuda:0', dtype=torch.float16,
       requires_grad=True)
tensor([[-0.0052,  0.0004,  0.0015,  ..., -0.0048,  0.0014, -0.0032],
        [ 0.0058, -0.0019, -0.0028,  ...,  0.0015, -0.0069, -0.0043],
        [ 0.0092,  0.0020,  0.0040,  ..., -0.0009,  0.0059,  0.0033],
        ...,
        [ 0.0029, -0.0078,  0.0033,  ..., -0.0019, -0.0015,  0.0056],
        [-0.0015, -0.0005, -0.0031,  ...,  0.0031, -0.0021, -0.0014],
        [-0.0016,  0.0030,  0.0032,  ...,  0.0014,  0.0008, -0.0012]],
       device='cuda:0', dtype=torch.float16, grad_fn=<MulBackward0>)
Parameter containing:
tensor([[-0.0052,  0.0004,  0.0015,  ..., -0.0048,  0.0014, -0.0032],
        [ 0.0058, -0.0019, -0.0028,  ...,  0.0015, -0.0069, -0.0043],
        [ 0.0092,  0.0020,  0.0040,  ..., -0.0009,  0.0059,

In [33]:
down_proj_weight = model.model.layers[0].mlp.down_proj.weight
down_proj_weight_fp16 = flh_fp16_model.model.layers[0].mlp.down_proj.weight

print(down_proj_weight)
print(down_proj_weight_fp16)

down_proj_weight_quant = flh_model.model.layers[0].mlp.down_proj.weight
down_proj_weight_quant = flh.nn.fast_hadamard_transform(down_proj_weight_quant, group_size=128, normalize=True)
down_proj_weight_quant = flh.nn.fast_hadamard_transform(down_proj_weight_quant.t(), group_size=128, normalize=False).t()

print(down_proj_weight_quant)

Parameter containing:
tensor([[-0.0286,  0.0173,  0.0011,  ..., -0.0203,  0.0137,  0.0067],
        [ 0.0190, -0.0053,  0.0080,  ..., -0.0256, -0.0006,  0.0048],
        [ 0.0243, -0.0071,  0.0278,  ...,  0.0168, -0.0033,  0.0013],
        ...,
        [ 0.0012, -0.0175, -0.0086,  ...,  0.0004, -0.0047,  0.0045],
        [-0.0004, -0.0311, -0.0068,  ..., -0.0306, -0.0270,  0.0082],
        [ 0.0383, -0.0040,  0.0063,  ...,  0.0074,  0.0076, -0.0156]],
       device='cuda:0', dtype=torch.float16, requires_grad=True)
Parameter containing:
tensor([[-0.0286,  0.0173,  0.0011,  ..., -0.0203,  0.0137,  0.0067],
        [ 0.0190, -0.0053,  0.0080,  ..., -0.0256, -0.0006,  0.0048],
        [ 0.0243, -0.0071,  0.0278,  ...,  0.0168, -0.0033,  0.0013],
        ...,
        [ 0.0012, -0.0175, -0.0086,  ...,  0.0004, -0.0047,  0.0045],
        [-0.0004, -0.0311, -0.0068,  ..., -0.0306, -0.0270,  0.0082],
        [ 0.0383, -0.0040,  0.0063,  ...,  0.0074,  0.0076, -0.0156]],
       device='cuda:0',

In [37]:
print(model.model.norm.weight)
print(flh_fp16_model.model.norm.weight)
print(flh_model.model.norm.weight)

Parameter containing:
tensor([2.4688, 2.2812, 1.5078,  ..., 2.5156, 2.4062, 2.5000], device='cuda:2',
       dtype=torch.float16, requires_grad=True)
Parameter containing:
tensor([2.4688, 2.2812, 1.5078,  ..., 2.5156, 2.4062, 2.5000], device='cuda:0',
       dtype=torch.float16, requires_grad=True)
Parameter containing:
tensor([2.4688, 2.2812, 1.5078,  ..., 2.5156, 2.4062, 2.5000], device='cuda:0',
       dtype=torch.float16, requires_grad=True)
